Karisma AI — Model 1: Skill Extractor

**Capstone Project CC26-PSU202**

## Arsitektur & Strategi

Model ini bertugas mengekstrak skill dari raw text CV (bahasa Inggris).

**Pendekatan: Hybrid — DistilBERT Fine-tuned + Custom Callback**

| Komponen | Detail |
|---|---|
| Base Model | `distilbert-base-uncased` (pre-trained, di-fine-tune) |
| Task | Token Classification / NER (BIO tagging) |
| Label | B-SKILL, I-SKILL, O |
| Custom Component | `SkillExtractionCallback` (Custom Callback) |
| Dataset | `glints_nlp_ready.csv` → skill vocabulary → synthetic CV sentences |
| Output | Model `.keras` / SavedModel siap produksi |

**Kenapa Synthetic Sentences?**
> Dataset Glints berisi daftar skill per job posting, bukan kalimat CV. Kita generate synthetic CV sentences menggunakan template realistis, lalu label skill-nya secara otomatis dengan BIO tagging.

**Alur Inference (Production):**
```
PDF CV → raw text → Model 1 → ['python', 'sql', 'machine learning'] → Model 2 → Top 3 Careers
```

## Cell 1 — Install Dependencies

In [1]:
!pip install transformers==4.44.0 --quiet
!pip install tensorflow-hub --quiet
!pip install seqeval

import os
print('Dependencies installed')

Dependencies installed


## Cell 2 — Mount Google Drive & Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Lokasi dataset di Google Drive
DATASET_PATH = '/content/drive/MyDrive/karisma-ai/glints_nlp_ready.csv'
# Lokasi log di Google Drive
DRIVE_LOG_DIR = '/content/drive/MyDrive/karisma-ai/logs/model1_skill_extractor'

import pandas as pd
df = pd.read_csv(DATASET_PATH)
print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

## Cell 3 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import json
import random
from collections import Counter

import tensorflow as tf
from transformers import (
    DistilBertTokenizerFast,
    TFDistilBertModel,
    DistilBertConfig
)
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score, accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(gpus) > 0} — {gpus}')

## Cell 4 — EDA: Eksplorasi Dataset & Skill Vocabulary

In [ ]:
# ── Parse skills_list dari string ke Python list ──
def parse_skills(s):
    try:
        skills = ast.literal_eval(s)
        return [sk.lower().strip() for sk in skills if sk.strip()]
    except:
        return []

df['skills_parsed'] = df['skills_list'].apply(parse_skills)

# ── Build skill vocabulary ──
all_skills = []
for skills in df['skills_parsed']:
    all_skills.extend(skills)

skill_counter = Counter(all_skills)
# Filter: minimal muncul 2x untuk menghindari typo/noise
SKILL_VOCAB = sorted([skill for skill, count in skill_counter.items() if count >= 2])

print(f'Total rows: {len(df):,}')
print(f'Total skill mentions: {len(all_skills):,}')
print(f'Unique skills (raw): {len(skill_counter):,}')
print(f'Skill vocabulary (freq >= 2): {len(SKILL_VOCAB):,}')
print()

# ── Visualisasi distribusi ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Top 20 skills
top20 = pd.Series(all_skills).value_counts().head(20)
axes[0].barh(top20.index[::-1], top20.values[::-1], color='steelblue')
axes[0].set_title('Top 20 Most Common Skills', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Frequency')

# Distribusi panjang skill (jumlah kata)
skill_lengths = [len(s.split()) for s in SKILL_VOCAB]
length_dist = Counter(skill_lengths)
axes[1].bar(length_dist.keys(), length_dist.values(), color='coral')
axes[1].set_title('Skill Length Distribution (words)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Words in Skill')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_skill_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA plots saved')

## Cell 5 — Bangun Dataset Training: Synthetic CV Sentences + BIO Tagging

> **Strategi:** Karena dataset Glints adalah job posting (bukan CV), kita buat synthetic training sentences menggunakan template CV yang realistis. Skill dari vocabulary di-embed ke dalam kalimat, lalu di-tag dengan format BIO (B-SKILL, I-SKILL, O).
>
> Ini adalah teknik **weak supervision / programmatic labeling** yang umum di industri NLP.

In [ ]:
"""
Generate Synthetic Training Data
Perbaikan:
  1. Lebih banyak variasi skill templates (action-result, bullet)
  2. Negative templates: kontak, pendidikan, header ALL CAPS,
     format PDF berantakan, baris kosong, award, org
  3. Data dummy Indonesia-aware (nama, universitas lokal)
  4. 40.000 sampel: 60% skill, 40% negative
  5. EKSPANSI MULTI-DISIPLIN: Ditambahkan ratusan data dummy
     dan template Non-IT (Bisnis, Kesehatan, Teknik, HR, Desain)
  6. ANTI-LINK: Penambahan berbagai format URL dan link agar
     tidak diekstrak sebagai skill.
"""


# ── SKILL TEMPLATES ──
SKILL_TEMPLATES = [
    # Pernyataan skill langsung
    "Proficient in {skill1} and {skill2} with hands-on experience.",
    "Strong background in {skill1}, {skill2}, and {skill3}.",
    "I have experience in {skill1} and solid knowledge of {skill2}.",
    "Skills include {skill1}, {skill2}, {skill3}, and {skill4}.",
    "Experienced professional with expertise in {skill1} and {skill2}.",
    "Developed and maintained projects using {skill1} and {skill2}.",
    "Responsible for {skill1}, {skill2}, and {skill3} in previous role.",
    "Technical skills: {skill1}, {skill2}, {skill3}.",
    "Key competencies: {skill1} and {skill2}.",
    "Worked with {skill1} to deliver results, also knowledgeable in {skill2}.",
    "Demonstrated ability in {skill1} and {skill2} throughout career.",
    "Core skills: {skill1}, {skill2}, {skill3}, and {skill4}.",
    "Used {skill1} daily. Also experienced in {skill2} and {skill3}.",
    "Certified in {skill1} with additional skills in {skill2}.",
    "Applied {skill1} and {skill2} to complete projects on time and budget.",
    "My technical toolkit includes {skill1}, {skill2}, and {skill3}.",
    "Fluent in {skill1} and comfortable working with {skill2}.",
    "Led teams using {skill1} methodology and managed {skill2} tools.",
    "Collaborated cross-functionally, utilizing {skill1} and {skill2} skills.",
    "Advanced knowledge of {skill1}, with working experience in {skill2} and {skill3}.",
    "Hands-on experience building systems with {skill1} and deploying using {skill2}.",
    "Coursework included {skill1}, {skill2}, and {skill3}.",
    "Personal projects demonstrate proficiency in {skill1} and {skill2}.",
    "Adept at {skill1}, frequently applying {skill2} in day-to-day tasks.",
    "Delivered client solutions leveraging {skill1}, {skill2}, and {skill3}.",
    "Improved team efficiency by implementing {skill1} and {skill2}.",
    "Completed certification in {skill1}; also skilled in {skill2}.",
    "Strong hands-on skills in {skill1} and {skill2} from internship experience.",

    # Ekstraksi Non-IT / General Roles (Bisnis, Kreatif, Kesehatan, dll)
    "Directed successful campaigns utilizing {skill1} and {skill2}.",
    "Evaluated financial risks using {skill1} principles and {skill2}.",
    "Provided high-quality patient care relying on {skill1} and {skill2}.",
    "Created engaging visual assets with {skill1} and {skill2}.",
    "Streamlined HR processes leveraging {skill1} and {skill2}.",
    "Conducted market research applying {skill1} and {skill2} methodologies.",
    "Managed daily operations ensuring {skill1} and {skill2} standards are met.",

    # Bullet point style — model belajar bullet itu O, skill-nya B-SKILL
    "• {skill1}, {skill2}, and {skill3}.",
    "• {skill1} and {skill2}.",
    "- {skill1}, {skill2}, {skill3}.",
    "- Experienced in {skill1} and {skill2}.",
    "• Developed backend services using {skill1} and {skill2}.",
    "• Built REST APIs with {skill1} and connected to {skill2} database.",
    "• Implemented {skill1} architecture to improve system performance.",
    "• Migrated legacy codebase to {skill1}, improving maintainability.",
    "• Collaborated with team using {skill1} and {skill2} workflows.",
    "• Integrated {skill1} with {skill2} for seamless data pipeline.",
    "- Designed UI components using {skill1} and {skill2}.",
    "- Automated deployment pipeline using {skill1} and {skill2}.",
    "• Executed sales strategies utilizing {skill1} and {skill2}.",
    "• Handled bookkeeping and audit preparation using {skill1}.",
    "• Maintained safety compliance according to {skill1} protocols.",

    # Action-Result style (gaya CV profesional dengan metrik)
    "Increased system performance by 30% by implementing {skill1} and {skill2}.",
    "Reduced latency by 40% through optimizing {skill1} queries.",
    "Delivered a full-stack application using {skill1}, {skill2}, and {skill3} within 3 months.",
    "Managed a team of 5 engineers to build a {skill1}-based platform.",
    "Spearheaded integration of {skill1} with {skill2} for real-time data processing.",
    "Automated reporting workflow using {skill1}, saving 10 hours per week.",
    "Improved user retention by 25% by redesigning frontend with {skill1}.",
    "Trained 3 junior developers on {skill1} and {skill2} best practices.",
    "Built and deployed microservices using {skill1} and {skill2} on cloud infrastructure.",
    "Refactored legacy code using {skill1}, reducing bug count by 60%.",
    "Designed relational database schema using {skill1} to support 100k+ users.",
    "Implemented authentication system using {skill1} and {skill2}.",
    "Created data visualization dashboards using {skill1} and {skill2}.",
    "Conducted A/B testing using {skill1} to optimize conversion rates.",
    "Developed mobile application using {skill1} with {skill2} for state management.",
    "Grew social media engagement by 50% using {skill1} and {skill2} strategies.",
    "Reduced operational costs by 15% through {skill1} and careful {skill2}.",
    "Closed $500k in B2B sales by leveraging {skill1} and {skill2} techniques."
]

# ── NEGATIVE TEMPLATES ──
# Model harus belajar bahwa isi template ini BUKAN skill
NEGATIVE_TEMPLATES = [
    # Header kontak
    "{name} | {email} | {phone} | {city}, Indonesia",
    "Name: {name}. Email: {email}. Phone: {phone}.",
    "Contact: {email} — LinkedIn: linkedin.com/in/{username}",
    "GitHub: github.com/{username} | Portfolio: {username}.dev",
    "{city}, Indonesia | {phone} | {email}",
    "linkedin.com/in/{username}-{code} | github.com/{username}",
    "{name} \\ {city} \\ {email} \\ {phone}",

    # Berbagai format URL dan Link
    "https://github.com/{username}",
    "http://www.linkedin.com/in/{username}-{code}",
    "Portfolio: {username}.github.io",
    "Website: www.{username}.com",
    "Medium: medium.com/@{username}",
    "Behance: behance.net/{username}",
    "Dribbble: dribbble.com/{username}",
    "{email} | github.com/{username} | linkedin.com/in/{username}",

    # Pendidikan
    "{university}, {city} — {degree} in {major}, GPA {gpa}, {year}",
    "Bachelor of {major}, {university}. Graduated {year}. GPA: {gpa}/4.00.",
    "Studied {major} at {university} from {year} to {year2}.",
    "{degree}, {major} — {university} ({year}–{year2}). GPA {gpa}.",
    "Expected graduation: {year2}. Currently enrolled at {university}.",
    "High School: {school}, {city}. Graduated {year}.",
    "GPA: {gpa} / 4.00",
    "GPA {gpa}/4.00 | {university} | {year} - {year2}",

    # Pengalaman kerja (konteks, tanpa skill)
    "{company}, {city} — {jobtitle} ({month} {year} – {month2} {year2})",
    "Worked at {company} as a {jobtitle} for {duration} months.",
    "Internship at {company}, {city}, from {month} to {month2} {year}.",
    "{jobtitle} | {company} | {year} – Present",
    "Part-time {jobtitle} at {company} during {year}–{year2}.",

    # Organisasi & penghargaan
    "Member of {org} since {year}.",
    "Winner of {award} competition, {year}.",
    "Received {award} award from {university}.",
    "Active member: {org}.",
    "Speaker at {event}, {city}, {year}.",
    "• {award} - {year}",
    "• Vice President at {org}, {city}",
    "• Assisted in organizing {event} with 500+ participants.",
    "• Secretary of {org} ({year}–{year2})",
    "• Member of {org} and {org2}",

    # ALL CAPS section headers — sangat sering muncul di PDF parse
    "SUMMARY",
    "EDUCATION",
    "WORK EXPERIENCE",
    "TECHNICAL SKILLS",
    "HARD SKILLS",
    "SOFT SKILLS",
    "PROJECTS",
    "ORGANIZATIONAL EXPERIENCE",
    "CERTIFICATIONS",
    "ACHIEVEMENTS",
    "CONTACT INFORMATION",
    "EDUCATION AND QUALIFICATIONS",
    "PROJECTS AND PORTFOLIO",
    "REFERENCES",
    "LANGUAGES",

    # Format berantakan hasil PDF parse
    "{name}",
    "{email}",
    "{phone}",
    "{city}, Indonesia",
    "{gpa}/4.00",
    "Page 1 of 2",
    "Curriculum Vitae",
    "{name} — Curriculum Vitae",
    "{university} | {year}",
    "{month} {year} – {month2} {year2}",
    "{month} {year} – Present",
    "| {email} | {phone} |",

    # Mixed: kalimat panjang dengan 1-2 skill tapi banyak non-skill token
    "{name} is proficient in {skill1} and currently studying at {university}.",
    "As a {jobtitle} at {company}, I applied {skill1} daily.",
    "Graduated from {university} with a degree in {major}, skilled in {skill1}.",
    "GPA {gpa}/4.00 at {university}, with coursework in {skill1} and {skill2}.",
    "Interned at {company} in {city}, working on {skill1} projects.",
    "Contact me at {email} for more about my {skill1} experience.",
    "{name} graduated from {university} in {year} with expertise in {skill1}.",
    "Received {award} while studying {skill1} at {university}.",
    "As a member of {org}, I developed {skill1} skills alongside leadership experience.",
    "During my time at {company} ({year}–{year2}), I mainly used {skill1}.",
]

# ── Data dummy ──
NAMES = [
    "john smith", "sarah johnson", "michael lee", "jessica tan",
    "david chen", "amanda wilson", "ryan garcia", "emily nguyen",
    "christopher brown", "ashley davis", "matthew anderson", "olivia thomas",
    "daniel martinez", "sophia jackson", "james white", "isabella harris",
    # Indonesia-aware
    "josua situmorang", "budi santoso", "ayu lestari", "kevin sanjaya",
    "putri andini", "rizky pratama", "muhammad fadil", "agnes monica",
    "dian rahayu", "fajar nugroho", "siti nurhaliza", "andi firmansyah",
    "maya kusuma", "reza maulana", "indah permata", "bagas setiawan",
    "jonathan christian manik", "dian putri rahayu", "putri indah permatasari sitepu"
]
EMAILS = [
    "john.smith@gmail.com", "sarah.j@yahoo.com", "m.lee@outlook.com",
    "jessica.tan@hotmail.com", "davidchen@gmail.com", "a.wilson@email.com",
    "ryan.garcia28@gmail.com", "emily.nguyen@student.ac.id",
    "chris.brown99@gmail.com", "ashley.d@company.com",
    "yehezkiels.26052004@gmail.com", "budisantoso99@gmail.com",
    "rizky.pratama@student.usu.ac.id", "fadil.m@outlook.com",
    "info@creative-design.com", "finance.dept@company.com"
]
PHONES = [
    "+62)8111023149", "+62-812-3456-7890", "(+62) 896-1234-5678",
    "+1-555-234-5678", "+62 851 2345 6789", "08123456789",
    "+62)8234567890", "| +62)8111023149 |", "(+62)81234567890"
]
CITIES = [
    "medan", "jakarta", "bandung", "surabaya", "yogyakarta",
    "semarang", "makassar", "palembang", "denpasar", "malang",
    "bogor", "depok", "tangerang", "bekasi", "batam", "pekanbaru",
    "pontianak", "samarinda", "balikpapan", "manado"
]
USERNAMES = [
    "johnsmith", "sarahj", "mlee", "jessica.tan", "d.chen",
    "ryangarcia", "emily.n", "cbrown", "ashley.w", "matt.a",
    "yehezkiel-sibarani", "kielsitum", "budisantoso", "rizkyp",
    "designer_ayu", "nurse.budi", "sales_jessica", "hrd_fadil"
]
CODES = ["a127bb345", "b234cc456", "x098zz123", "p765qq890", "m312rr567"]
UNIVERSITIES = [
    "universitas sumatera utara", "universitas indonesia",
    "universitas gadjah mada", "institut teknologi bandung",
    "universitas brawijaya", "universitas diponegoro",
    "binus university", "universitas padjadjaran",
    "universitas hasanuddin", "universitas airlangga",
    "institut teknologi sepuluh nopember", "telkom university",
    "universitas sebelas maret", "politeknik negeri jakarta",
    "universitas mercu buana", "universitas tarumanagara",
    "universitas bina nusantara", "universitas pelita harapan",
    "politeknik keuangan negara stan", "sekolah tinggi ilmu ekonomi",
    "universitas udayana", "universitas negeri medan"
]
DEGREES = [
    "bachelor of science", "bachelor of engineering",
    "bachelor of computer science", "s.t.", "s.kom", "b.sc", "s.si",
    # Gelar Non-IT
    "bachelor of arts", "bachelor of business administration",
    "s.e.", "s.kep", "s.h.", "s.pd", "s.sos", "mba", "m.sc"
]
MAJORS = [
    "information technology", "computer science", "electrical engineering",
    "information systems", "software engineering", "data science",
    "mathematics", "physics", "management information systems",
    "informatics engineering", "computer engineering",
    # Jurusan Non-IT Terpopuler
    "accounting", "business administration", "psychology",
    "civil engineering", "mechanical engineering", "nursing",
    "public relations", "marketing", "law", "economics",
    "graphic design", "architecture", "communication studies", "medicine"
]
GPAS = ["3.79", "3.85", "3.62", "3.91", "3.45", "3.77", "3.88", "2.98", "3.55", "3.93", "3.21"]
YEARS = ["2019", "2020", "2021", "2022", "2023", "2024", "2025", "2026"]
MONTHS = ["january", "february", "march", "april", "may", "june",
          "july", "august", "september", "october", "november", "december"]
COMPANIES = [
    "google", "microsoft", "gojek", "tokopedia", "bukalapak",
    "traveloka", "shopee", "grab", "bank mandiri", "telkom indonesia",
    "pt. xyz solutions", "abc corp", "startup inc", "tech studio",
    "dbs foundation", "pt. digital nusantara", "cv. kreasi teknologi",
    # Perusahaan Non-IT Terkenal (FMCG, Bank, Konsultan, RS)
    "bank bri", "bank bca", "pertamina", "garuda indonesia",
    "unilever", "nestle", "pwc", "deloitte", "kpmg", "mckinsey",
    "siloam hospitals", "rsud dr. soetomo", "pt astra international",
    "indofood", "wings group", "kalbe farma"
]
JOBTITLES = [
    "software engineer intern", "backend developer", "frontend developer",
    "data analyst intern", "fullstack developer", "junior developer",
    "web developer", "mobile developer", "devops engineer",
    "it support intern", "ui/ux designer intern",
    # Posisi Non-IT Terpopuler
    "business analyst", "marketing manager", "financial analyst",
    "human resources specialist", "sales representative",
    "civil engineer", "registered nurse", "graphic designer",
    "project manager", "content writer", "accountant",
    "operations manager", "social media specialist", "legal counsel"
]
ORGS = [
    "student executive board", "coding club", "ieee student branch",
    "programming club", "robotics team", "math olympiad club",
    "himpunan mahasiswa informatika", "bem fakultas teknik",
    "dbs foundation coding camp", "google developer student club",
    # Organisasi General/Non-IT
    "himpunan mahasiswa akuntansi", "debate society",
    "business and investment club", "red cross youth",
    "student choir", "model united nations"
]
AWARDS = [
    "best student", "dean's list", "hackathon winner", "scholarship recipient",
    "best capstone project", "outstanding achievement",
    "first place", "best paper award", "academic excellence award",
    # Penghargaan Non-IT
    "employee of the month", "top sales performer",
    "best marketing campaign", "outstanding volunteer award"
]
EVENTS = [
    "tech conference", "developer summit", "student symposium",
    "national programming competition", "coding bootcamp",
    # Event Non-IT
    "national business case competition", "medical seminar",
    "creative design workshop", "leadership training program"
]
SCHOOLS = [
    "sma negeri 1 medan", "sma plus pembangunan", "sman 3 bandung",
    "sma methodist 2", "sman 5 surabaya", "sma santa ursula",
    "smk negeri 1 akuntansi", "smk telkom", "smk negeri 7"
]
DURATIONS = ["3", "4", "6", "8", "12", "18", "24"]

def fill_negative_template(template, skill_vocab, rng):
    result = template
    skill_slots = re.findall(r'\{skill\d+\}', template)
    used_skills = []
    for slot in skill_slots:
        sk = rng.choice(skill_vocab)
        result = result.replace(slot, sk, 1)
        used_skills.append(sk)

    replacements = {
        '{name}'      : rng.choice(NAMES),
        '{email}'     : rng.choice(EMAILS),
        '{phone}'     : rng.choice(PHONES),
        '{city}'      : rng.choice(CITIES),
        '{username}'  : rng.choice(USERNAMES),
        '{code}'      : rng.choice(CODES),
        '{university}': rng.choice(UNIVERSITIES),
        '{degree}'    : rng.choice(DEGREES),
        '{major}'     : rng.choice(MAJORS),
        '{gpa}'       : rng.choice(GPAS),
        '{year}'      : rng.choice(YEARS),
        '{year2}'     : rng.choice(YEARS),
        '{month}'     : rng.choice(MONTHS),
        '{month2}'    : rng.choice(MONTHS),
        '{company}'   : rng.choice(COMPANIES),
        '{jobtitle}'  : rng.choice(JOBTITLES),
        '{org}'       : rng.choice(ORGS),
        '{org2}'      : rng.choice(ORGS),
        '{award}'     : rng.choice(AWARDS),
        '{event}'     : rng.choice(EVENTS),
        '{school}'    : rng.choice(SCHOOLS),
        '{duration}'  : rng.choice(DURATIONS),
    }
    for placeholder, value in replacements.items():
        result = result.replace(placeholder, value)
    return result, used_skills

def fill_skill_template(template, skill_vocab, rng):
    placeholders = re.findall(r'\{skill\d+\}', template)
    n_needed = len(placeholders)
    selected = rng.sample(skill_vocab, min(n_needed, len(skill_vocab)))
    filled = template
    for i, ph in enumerate(placeholders):
        filled = filled.replace(ph, selected[i], 1)
    return filled, selected

def bio_tag_sentence(sentence, skills_in_sentence):
    skills_sorted = sorted(skills_in_sentence, key=lambda s: len(s.split()), reverse=True)
    tokens = sentence.split()
    tags = ['O'] * len(tokens)
    for skill in skills_sorted:
        skill_tokens = skill.split()
        n = len(skill_tokens)
        for i in range(len(tokens) - n + 1):
            window = [t.lower().strip('.,;:()"\'•-') for t in tokens[i:i+n]]
            if window == skill_tokens:
                if all(tags[i+j] == 'O' for j in range(n)):
                    tags[i] = 'B-SKILL'
                    for j in range(1, n):
                        tags[i+j] = 'I-SKILL'
    return tokens, tags

# ── Generate ──
rng = random.Random(SEED) # Pastikan SEED sudah di-define di cell sebelumnya

EN_SKILL_VOCAB = [
    s for s in SKILL_VOCAB
    if re.match(r'^[a-zA-Z0-9 .#+/&-]+$', s) and len(s) <= 40
]
print(f'English-compatible skills: {len(EN_SKILL_VOCAB):,}')

N_SKILL_SAMPLES    = 24000
N_NEGATIVE_SAMPLES = 16000

training_data = []

for _ in range(N_SKILL_SAMPLES):
    template = rng.choice(SKILL_TEMPLATES)
    sentence, skills_used = fill_skill_template(template, EN_SKILL_VOCAB, rng)
    tokens, tags = bio_tag_sentence(sentence, skills_used)
    training_data.append((tokens, tags))

for _ in range(N_NEGATIVE_SAMPLES):
    template = rng.choice(NEGATIVE_TEMPLATES)
    sentence, skills_used = fill_negative_template(template, EN_SKILL_VOCAB, rng)
    tokens, tags = bio_tag_sentence(sentence, skills_used)
    training_data.append((tokens, tags))

rng.shuffle(training_data)

print(f'Generated {len(training_data):,} sentences')
print(f'Skill sentences   : {N_SKILL_SAMPLES:,}')
print(f'Negative sentences: {N_NEGATIVE_SAMPLES:,}')

all_tags_flat = [t for _, tags in training_data for t in tags]
tag_dist = Counter(all_tags_flat)
print(f'\nLabel distribution:')
for tag, count in sorted(tag_dist.items()):
    pct = count / len(all_tags_flat) * 100
    print(f'  {tag:10s}: {count:,} ({pct:.1f}%)')

print('\nSample negative sentences (verifikasi):')
neg_samples = [x for x in training_data if all(t == 'O' for t in x[1])][:3]
for tokens, tags in neg_samples:
    print(f'  {" ".join(tokens)[:70]}')
    print(f'  tags: {tags[:8]}...')
    print()

## Cell 6 — Tokenisasi dengan DistilBERT Tokenizer + Align BIO Labels

In [ ]:
# ── Load DistilBERT Tokenizer ──
MODEL_CHECKPOINT = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_CHECKPOINT)

# ── Label mapping ──
LABEL2ID = {'O': 0, 'B-SKILL': 1, 'I-SKILL': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print(f'Label mapping: {LABEL2ID}')

# ── Fungsi align labels ke subword tokens ──
def tokenize_and_align_labels(tokens, tags, max_length=128):
    """
    DistilBERT memecah kata jadi subwords (e.g., 'learning' → ['learn', '##ing']).
    Label harus di-align: subword pertama dapat label asli, sisanya dapat -100 (ignored).
    """
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors=None
    )

    word_ids = encoding.word_ids()
    label_ids = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            # Special tokens [CLS], [SEP], [PAD]
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            # Token pertama dari setiap kata
            label_ids.append(LABEL2ID[tags[word_idx]])
        else:
            # Subword lanjutan: gunakan label sama tapi bisa juga -100
            # Kita pakai label sama (I-SKILL jika B-SKILL) agar model belajar span
            current_label = tags[word_idx]
            if current_label == 'B-SKILL':
                label_ids.append(LABEL2ID['I-SKILL'])
            else:
                label_ids.append(LABEL2ID[current_label])
        previous_word_idx = word_idx

    return {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': label_ids
    }

# ── Proses semua data ──
print('Tokenizing training data...')
MAX_LENGTH = 128

encoded_data = []
for tokens, tags in training_data:
    encoded = tokenize_and_align_labels(tokens, tags, max_length=MAX_LENGTH)
    encoded_data.append(encoded)

# Convert ke numpy arrays
input_ids = np.array([d['input_ids'] for d in encoded_data])
attention_masks = np.array([d['attention_mask'] for d in encoded_data])
labels = np.array([d['labels'] for d in encoded_data])

print(f'Tokenized {len(encoded_data):,} sentences')
print(f'input_ids shape: {input_ids.shape}')
print(f'attention_mask shape: {attention_masks.shape}')
print(f'labels shape: {labels.shape}')

## Cell 7 — Train/Val/Test Split

In [ ]:
# ── Split: 80% train, 10% val, 10% test ──
indices = np.arange(len(input_ids))

idx_train, idx_temp = train_test_split(indices, test_size=0.2, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=SEED)

X_train_ids = input_ids[idx_train]
X_train_masks = attention_masks[idx_train]
y_train = labels[idx_train]

X_val_ids = input_ids[idx_val]
X_val_masks = attention_masks[idx_val]
y_val = labels[idx_val]

X_test_ids = input_ids[idx_test]
X_test_masks = attention_masks[idx_test]
y_test = labels[idx_test]

print(f'Train set : {len(X_train_ids):,} samples')
print(f'Val set   : {len(X_val_ids):,} samples')
print(f'Test set  : {len(X_test_ids):,} samples')

## Cell 8 — Bangun Model: DistilBERT Fine-tuned (TF Functional API)

> Ini memenuhi **Main Quest**: model dibangun dengan TF/Keras, menggunakan DistilBERT sebagai backbone (pre-trained + fine-tuned), dan disimpan dalam format `.keras`.

In [ ]:
from transformers import TFDistilBertModel

# ── Bangun model dengan Keras Functional API ──
def build_skill_extractor_model(model_checkpoint, num_labels, max_length=128):
    """
    Arsitektur:
    Input → DistilBERT (pre-trained backbone) → Dropout → Dense (token classifier)

    Menggunakan TF Functional API sesuai main quest.
    """
    # Input layers
    input_ids_layer = tf.keras.Input(
        shape=(max_length,), dtype=tf.int32, name='input_ids'
    )
    attention_mask_layer = tf.keras.Input(
        shape=(max_length,), dtype=tf.int32, name='attention_mask'
    )

    # DistilBERT backbone (pre-trained)
    distilbert = TFDistilBertModel.from_pretrained(
        model_checkpoint,
        output_hidden_states=False
    )

    # Forward pass melalui DistilBERT
    # output.last_hidden_state shape: (batch, seq_len, 768)
    bert_output = distilbert(
        input_ids=input_ids_layer,
        attention_mask=attention_mask_layer
    )
    sequence_output = bert_output.last_hidden_state  # (batch, seq_len, 768)

    # Classification head
    dropout = tf.keras.layers.Dropout(0.1, name='classifier_dropout')(sequence_output)
    logits = tf.keras.layers.Dense(
        num_labels,
        kernel_initializer='glorot_uniform',
        name='classifier'
    )(dropout)  # (batch, seq_len, num_labels)

    model = tf.keras.Model(
        inputs=[input_ids_layer, attention_mask_layer],
        outputs=logits,
        name='karisma_skill_extractor'
    )

    return model, distilbert

model, distilbert_backbone = build_skill_extractor_model(
    MODEL_CHECKPOINT, NUM_LABELS, MAX_LENGTH
)

print('Model built with TF Functional API')
model.summary()

## Cell 9 — Custom Callback: SkillExtractionCallback

> Ini memenuhi **Main Quest**: implementasi Custom Callback.

In [ ]:
class SkillExtractionCallback(tf.keras.callbacks.Callback):
    """
    Custom Callback untuk Model 1 — Skill Extractor.

    Fungsi:
    1. Monitor val_loss dan simpan best model otomatis
    2. Early stopping jika val_loss tidak membaik
    3. Log ringkasan performa per epoch ke file JSON
    4. Tampilkan contoh prediksi skill setiap N epoch
    """

    def __init__(
        self,
        patience=3,
        min_delta=0.001,
        save_path='best_skill_extractor.keras',
        log_path='training_log.json',
        demo_texts=None,
        demo_every_n_epochs=2,
        target_val_loss=0.05
    ):
        super().__init__()
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.log_path = log_path
        self.demo_texts = demo_texts or [
            "Proficient in Python and machine learning with 3 years experience.",
            "• Managed a team of 5, specialized in digital marketing and graphic design.",
            "Contact: yehezkiel@gmail.com | GPA 3.79 | Universitas Sumatera Utara"
        ]
        self.demo_every_n_epochs = demo_every_n_epochs
        self.target_val_loss = target_val_loss

        self.best_val_loss = float('inf')
        self.wait = 0
        self.training_log = []
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_loss = logs.get('val_loss', float('inf'))
        val_acc = logs.get('val_accuracy', 0)

        # ── 1. Simpan log ──
        epoch_log = {
            'epoch': epoch + 1,
            'loss': round(logs.get('loss', 0), 4),
            'accuracy': round(logs.get('accuracy', 0), 4),
            'val_loss': round(val_loss, 4),
            'val_accuracy': round(val_acc, 4)
        }
        self.training_log.append(epoch_log)
        with open(self.log_path, 'w') as f:
            json.dump(self.training_log, f, indent=2)

        # ── 2. Save best model ──
        if val_loss < self.best_val_loss - self.min_delta:
            print(f'\n   val_loss improved {self.best_val_loss:.4f} → {val_loss:.4f}. Saving model...')
            self.best_val_loss = val_loss
            self.model.save(self.save_path)
            self.wait = 0
        else:
            self.wait += 1
            print(f'\n   val_loss tidak membaik ({self.wait}/{self.patience}). Best: {self.best_val_loss:.4f}')

        # ── 3. Early stopping ──
        if self.wait >= self.patience:
            self.stopped_epoch = epoch
            self.model.stop_training = True
            print(f'\n   Early stopping triggered at epoch {epoch + 1}.')
            print(f'   Best val_loss: {self.best_val_loss:.4f}')

        # ── 4. Demo prediksi ──
        if (epoch + 1) % self.demo_every_n_epochs == 0:
            print(f'\n   Demo Skill Extraction (Epoch {epoch + 1}):')
            self._demo_prediction()

        # ── 5. Notifikasi target tercapai ──
        if val_loss <= self.target_val_loss:
            print(f'\n   Target val_loss ({self.target_val_loss}) TERCAPAI!')

    def _demo_prediction(self):
        """Jalankan inference cepat pada demo texts untuk monitor kualitas."""
        for text in self.demo_texts:
            try:
                words = text.split()
                enc = tokenizer(
                    words, is_split_into_words=True,
                    return_tensors='tf', truncation=True,
                    padding='max_length', max_length=128
                )
                logits = self.model([
                    enc['input_ids'], enc['attention_mask']
                ], training=False)
                predictions = tf.argmax(logits, axis=-1).numpy()[0]
                word_ids = enc.word_ids()

                extracted = []
                current_skill = []
                prev_word_idx = None
                for i, wid in enumerate(word_ids):
                    if wid is None or wid == prev_word_idx:
                        continue
                    label = ID2LABEL[predictions[i]]
                    if label == 'B-SKILL':
                        if current_skill:
                            extracted.append(' '.join(current_skill))
                        current_skill = [words[wid].lower().strip('.,;:()"\'•- ')]
                    elif label == 'I-SKILL' and current_skill:
                        current_skill.append(words[wid].lower().strip('.,;:()"\'•- '))
                    else:
                        if current_skill:
                            extracted.append(' '.join(current_skill))
                        current_skill = []
                    prev_word_idx = wid
                if current_skill:
                    extracted.append(' '.join(current_skill))

                print(f'      Text: "{text[:60]}..."')
                print(f'      Skills found: {extracted}')
            except Exception as e:
                print(f'      Demo error: {e}')

    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0:
            print(f'\nTraining selesai dengan early stopping pada epoch {self.stopped_epoch + 1}')
        else:
            print('\nTraining selesai (semua epoch)')
        print(f'Training log disimpan ke: {self.log_path}')

print('SkillExtractionCallback (Custom Callback) defined')

## Cell 10 — Compile & Training Setup

In [ ]:
# ── Fine-tuning strategy: unfreeze semua layer ──
# DistilBERT akan di-fine-tune bersama classification head
distilbert_backbone.trainable = True

# ── Custom loss: Sparse Categorical Crossentropy dengan masking -100 ──
# Label -100 (special tokens & padding) harus di-ignore
class MaskedSparseCategoricalCrossentropy(tf.keras.losses.Loss):
    """
    Custom Loss yang mengabaikan token dengan label -100.
    Ini diperlukan karena BIO tagging menggunakan -100 untuk padding & special tokens.
    """
    def __init__(self, ignore_index=-100, **kwargs):
        super().__init__(**kwargs)
        self.ignore_index = ignore_index
        self.loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True, reduction='none'
        )

    def call(self, y_true, y_pred):
        # Cast labels
        y_true = tf.cast(y_true, tf.int32)

        # Mask: True di mana label bukan -100
        mask = tf.not_equal(y_true, self.ignore_index)

        # Replace -100 dengan 0 untuk menghindari error di loss fn
        y_true_clean = tf.where(mask, y_true, tf.zeros_like(y_true))

        # Hitung loss
        losses = self.loss_fn(y_true_clean, y_pred)

        # Apply mask: hanya hitung loss pada token yang valid
        mask_float = tf.cast(mask, tf.float32)
        masked_losses = losses * mask_float

        # Mean over valid tokens
        return tf.reduce_sum(masked_losses) / (tf.reduce_sum(mask_float) + 1e-8)

    def get_config(self):
        config = super().get_config()
        config.update({'ignore_index': self.ignore_index})
        return config

# ── Custom Accuracy Metric dengan masking ──
class MaskedAccuracy(tf.keras.metrics.Metric):
    def __init__(self, ignore_index=-100, name='masked_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)
        self.ignore_index = ignore_index
        self.correct = self.add_weight(name='correct', initializer='zeros')
        self.total = self.add_weight(name='total', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int32)
        mask = tf.not_equal(y_true, self.ignore_index)

        pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        correct = tf.equal(y_true, pred_ids)
        correct_masked = tf.logical_and(correct, mask)

        self.correct.assign_add(tf.reduce_sum(tf.cast(correct_masked, tf.float32)))
        self.total.assign_add(tf.reduce_sum(tf.cast(mask, tf.float32)))

    def result(self):
        return self.correct / (self.total + 1e-8)

    def reset_state(self):
        self.correct.assign(0)
        self.total.assign(0)

# ── Learning rate schedule: warm-up + decay ──
BATCH_SIZE = 16
EPOCHS = 10
STEPS_PER_EPOCH = len(X_train_ids) // BATCH_SIZE
WARMUP_STEPS = STEPS_PER_EPOCH * 1  # 1 epoch warmup
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=5e-5,
    decay_steps=TOTAL_STEPS,
    end_learning_rate=1e-6
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# ── Compile ──
model.compile(
    optimizer=optimizer,
    loss=MaskedSparseCategoricalCrossentropy(name='masked_ce_loss'),
    metrics=[MaskedAccuracy(name='accuracy')]
)

print('Model compiled')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Steps per epoch: {STEPS_PER_EPOCH}')
print(f'Total parameters: {model.count_params():,}')

## Cell 11 — Training

In [ ]:
# ── Buat tf.data.Dataset untuk efisiensi ──
def make_dataset(ids, masks, lbls, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((
        {'input_ids': ids, 'attention_mask': masks},
        lbls
    ))
    if shuffle:
        ds = ds.shuffle(buffer_size=5000, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(X_train_ids, X_train_masks, y_train, BATCH_SIZE, shuffle=True)
val_ds = make_dataset(X_val_ids, X_val_masks, y_val, BATCH_SIZE)
test_ds = make_dataset(X_test_ids, X_test_masks, y_test, BATCH_SIZE)

# ── Custom callback instance ──
demo_texts = [
    "I am proficient in Python, machine learning, and data analysis.",
    "• Managed a team of 5, specialized in digital marketing and graphic design.",
    "Contact: yehezkiel@gmail.com | GPA 3.79 | Universitas Sumatera Utara"
]

skill_callback = SkillExtractionCallback(
    patience=3,
    min_delta=0.001,
    save_path='best_skill_extractor.keras',
    log_path='training_log.json',
    demo_texts=demo_texts,
    demo_every_n_epochs=2,
    target_val_loss=0.05
)

# TensorBoard callback (Side Quest)
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=DRIVE_LOG_DIR,
    histogram_freq=1,
    write_graph=True
)

print('Memulai training...')
print(f'Train samples: {len(X_train_ids):,}')
print(f'Val samples  : {len(X_val_ids):,}')
print()

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[skill_callback, tensorboard_callback],
    verbose=1
)

print('\nTraining selesai!')

In [ ]:
# Memuat ekstensi TensorBoard
%load_ext tensorboard

# Panggil dari Google Drive
%tensorboard --logdir /content/drive/MyDrive/karisma-ai/logs/model1_skill_extractor

## Cell 12 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history.history['loss']) + 1)

# Loss plot
axes[0].plot(epochs_ran, history.history['loss'], 'b-o', label='Train Loss')
axes[0].plot(epochs_ran, history.history['val_loss'], 'r-o', label='Val Loss')
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(epochs_ran, history.history['accuracy'], 'b-o', label='Train Accuracy')
axes[1].plot(epochs_ran, history.history['val_accuracy'], 'r-o', label='Val Accuracy')
axes[1].axhline(y=0.85, color='green', linestyle='--', label='Target (85%)')
axes[1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=120, bbox_inches='tight')
plt.show()

best_val_acc = max(history.history['val_accuracy'])
best_val_loss = min(history.history['val_loss'])
print(f'Best val_accuracy : {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'Best val_loss     : {best_val_loss:.4f}')
print(f'Target 85%        : {"TERCAPAI" if best_val_acc >= 0.85 else "Belum tercapai"}')

## Cell 13 — Evaluasi Detail: seqeval F1 Score per Label

In [ ]:
# ── Load best model ──
print('Loading best model...')
best_model = tf.keras.models.load_model(
    'best_skill_extractor.keras',
    custom_objects={
        'MaskedSparseCategoricalCrossentropy': MaskedSparseCategoricalCrossentropy,
        'MaskedAccuracy': MaskedAccuracy,
        'TFDistilBertModel': TFDistilBertModel
    }
)

# ── Prediksi pada test set ──
print('Running predictions on test set...')
all_true_labels = []
all_pred_labels = []

for batch_x, batch_y in test_ds:
    logits = best_model(batch_x, training=False)
    predictions = tf.argmax(logits, axis=-1).numpy()
    true_labels = batch_y.numpy()

    for pred_seq, true_seq in zip(predictions, true_labels):
        pred_labels_row = []
        true_labels_row = []
        for p, t in zip(pred_seq, true_seq):
            if t == -100:
                continue  # skip special tokens & padding
            pred_labels_row.append(ID2LABEL[p])
            true_labels_row.append(ID2LABEL[t])
        all_pred_labels.append(pred_labels_row)
        all_true_labels.append(true_labels_row)

# ── seqeval report ──
print('\n' + '='*60)
print('CLASSIFICATION REPORT (seqeval — entity-level)')
print('='*60)
print(classification_report(all_true_labels, all_pred_labels))

f1 = f1_score(all_true_labels, all_pred_labels)
print(f'Entity-level F1 Score: {f1:.4f} ({f1*100:.2f}%)')

## Cell 14 — Simpan Model dalam Format Produksi (.keras)

In [ ]:
import os

# ── Simpan model .keras (main quest) ──
FINAL_MODEL_PATH = 'karisma_skill_extractor_v1.keras'
best_model.save(FINAL_MODEL_PATH)
print(f'Model saved: {FINAL_MODEL_PATH}')
print(f'File size: {os.path.getsize(FINAL_MODEL_PATH) / 1024 / 1024:.1f} MB')

# ── Simpan juga sebagai SavedModel (opsional, untuk serving) ──
SAVEDMODEL_PATH = '/content/drive/MyDrive/karisma-ai/model1/karisma_skill_extractor_savedmodel'
best_model.save(SAVEDMODEL_PATH)
print(f'SavedModel saved: {SAVEDMODEL_PATH}/')

# ── Simpan skill vocabulary (dibutuhkan saat inference) ──
vocab_data = {
    'skill_vocab': EN_SKILL_VOCAB,
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'model_checkpoint': MODEL_CHECKPOINT,
    'max_length': MAX_LENGTH,
    'total_skills': len(EN_SKILL_VOCAB)
}

with open('skill_vocab.json', 'w') as f:
    json.dump(vocab_data, f, indent=2)
print(f'Skill vocabulary saved: skill_vocab.json ({len(EN_SKILL_VOCAB):,} skills)')

# ── Copy ke Google Drive ──
import shutil
DRIVE_SAVE_DIR = '/content/drive/MyDrive/karisma-ai/model1/'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy(FINAL_MODEL_PATH, DRIVE_SAVE_DIR)
shutil.copy('skill_vocab.json', DRIVE_SAVE_DIR)
shutil.copy('training_log.json', DRIVE_SAVE_DIR)
shutil.copy('training_history.png', DRIVE_SAVE_DIR)
shutil.copy('eda_skill_distribution.png', DRIVE_SAVE_DIR)
print(f'\nSemua file disimpan ke Google Drive: {DRIVE_SAVE_DIR}')

## Cell 15 — Inference: Ekstrak Skill dari CV Text

> Ini memenuhi **Main Quest**: kode inference model.

In [ ]:
class SkillExtractor:
    """
    Production inference class untuk Model 1 — Skill Extractor.

    Cara kerja:
    1. Terima raw text CV (string)
    2. Tokenize per kalimat
    3. Jalankan DistilBERT NER
    4. Kumpulkan span B-SKILL / I-SKILL
    5. Return list of skills
    """

    def __init__(self, model_path, vocab_path, tokenizer_name='distilbert-base-uncased'):
        print('Loading SkillExtractor...')

        # Load model
        self.model = tf.keras.models.load_model(
            model_path,
            custom_objects={
                'MaskedSparseCategoricalCrossentropy': MaskedSparseCategoricalCrossentropy,
                'MaskedAccuracy': MaskedAccuracy,
                'TFDistilBertModel': TFDistilBertModel
            }
        )

        # Load tokenizer
        self.tokenizer = DistilBertTokenizerFast.from_pretrained(tokenizer_name)

        # Load vocab & config
        with open(vocab_path) as f:
            vocab_data = json.load(f)
        self.id2label = {int(k): v for k, v in vocab_data['id2label'].items()}
        self.max_length = vocab_data['max_length']

        print(f'SkillExtractor ready')

    def _split_sentences(self, text):
        import re
        """
        Preprocessing + split raw text CV dari PDF parse.
        Menangani: word-wrap, soft-wrap, noise, dan single-word skills.
        """
        # --- Step 1: Gabungkan hyphenation di akhir baris ---
        lines = text.split('\n')
        merged_lines = []
        i = 0
        while i < len(lines):
            line = lines[i]
            if (re.search(r'[a-zA-Z]-$', line.rstrip()) and
                    i + 1 < len(lines) and
                    lines[i + 1].strip() and
                    not lines[i + 1].strip().startswith(('-', '•', '*'))):
                merged_lines.append(line.rstrip() + lines[i + 1].strip())
                i += 2
            else:
                merged_lines.append(line)
                i += 1

        # --- Step 2: Gabungkan soft-wrap (kalimat terpotong newline) ---
        rejoined = []
        i = 0
        while i < len(merged_lines):
            current = merged_lines[i].strip()
            if not current:
                rejoined.append('')
                i += 1
                continue
            while i + 1 < len(merged_lines):
                next_line = merged_lines[i + 1].strip()
                ends_no_punct = current and current[-1] not in '.!?:'
                next_lowercase = next_line and next_line[0].islower()
                not_header = not re.match(
                    r'^(january|february|march|april|may|june|july|august|'
                    r'september|october|november|december|present)',
                    current, re.IGNORECASE
                )
                if ends_no_punct and next_lowercase and not_header and next_line:
                    current = current + ' ' + next_line
                    i += 1
                else:
                    break
            rejoined.append(current)
            i += 1

        # --- Step 3: Split ke sentences, filter noise, SELAMATKAN SINGLE SKILL ---
        sentences = []
        for line in rejoined:
            line = line.strip()
            if not line:
                continue

            # Filter noise (URL, email, telp, hal, kurung)
            if re.match(r'^https?://', line) or re.match(r'^www\.', line): continue
            if re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', line): continue
            if re.match(r'^[\+\(]?\d[\d\s\-\(\)\.]{6,}$', line): continue
            if re.match(r'^Page \d+ of \d+$', line, re.IGNORECASE): continue
            if re.match(r'^\(.*\)$', line): continue

            # Bersihkan bullet markers
            line = re.sub(r'^[\•\-\*]\s+', '', line).strip()
            if not line: continue

            # Kalimat panjang: split per sentence ending
            if len(line.split()) > 25:
                subs = re.split(r'(?<=[.!?])\s+', line)
                for s in subs:
                    s = s.strip()
                    if s: sentences.append(s)
            else:
                word_count = len(line.split())
                if word_count > 0 and word_count <= 3:
                    sentences.append(f"Skill: {line}")
                else:
                    sentences.append(line)

        return sentences

    def _predict_sentence(self, sentence):
        """Prediksi BIO tags untuk satu kalimat."""
        words = sentence.split()
        if not words:
            return []

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            return_tensors='tf',
            truncation=True,
            padding='max_length',
            max_length=self.max_length
        )

        logits = self.model(
            [encoding['input_ids'], encoding['attention_mask']],
            training=False
        )
        predictions = tf.argmax(logits, axis=-1).numpy()[0]
        word_ids = encoding.word_ids()

        # Collect skills dari BIO tags
        skills = []
        current_skill_tokens = []
        prev_word_idx = None

        for i, word_idx in enumerate(word_ids):
            if word_idx is None or word_idx == prev_word_idx:
                continue

            label = self.id2label[predictions[i]]
            word = words[word_idx].lower().strip('.,;:()"\'•- ')

            if label == 'B-SKILL':
                if current_skill_tokens:
                    skills.append(' '.join(current_skill_tokens))
                current_skill_tokens = [word] if word else []
            elif label == 'I-SKILL' and current_skill_tokens:
                if word:
                    current_skill_tokens.append(word)
            else:
                if current_skill_tokens:
                    skills.append(' '.join(current_skill_tokens))
                current_skill_tokens = []

            prev_word_idx = word_idx

        if current_skill_tokens:
            skills.append(' '.join(current_skill_tokens))

        return skills

    def extract(self, cv_text, deduplicate=True):
        """
        Main method: ekstrak skill dari raw CV text.

        Args:
            cv_text (str): Raw text dari CV (hasil parsing PDF)
            deduplicate (bool): Hapus duplikat skill

        Returns:
            list: Daftar skill yang ditemukan
        """
        cv_text = re.sub(r'https?://\S+|www\.\S+', '', cv_text)
        cv_text = re.sub(r'(github\.com|linkedin\.com)(/\S+)?', '', cv_text)
        cv_text = re.sub(r'[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(/\S+)?', '', cv_text)

        sentences = self._split_sentences(cv_text)

        all_skills = []
        for sentence in sentences:
            skills = self._predict_sentence(sentence)
            all_skills.extend(skills)

        if deduplicate:
            # Deduplicate, preserve order
            seen = set()
            unique_skills = []
            for s in all_skills:
                if s not in seen and len(s) > 1:
                    seen.add(s)
                    unique_skills.append(s)
            return unique_skills

        return all_skills


# ── Inisialisasi SkillExtractor ──
extractor = SkillExtractor(
    model_path='karisma_skill_extractor_v1.keras',
    vocab_path='skill_vocab.json'
)

print('\n' + '='*60)
print('DEMO INFERENCE — Skill Extraction dari CV Text')
print('='*60)

# ── Demo 1: CV Data Scientist ──
cv_text_1 = """
I am a data scientist with 3 years of experience in machine learning and data analysis.
Proficient in Python, SQL, and TensorFlow. Strong background in data visualization using
Tableau and Power BI. Experience with deep learning, NLP, and statistical modeling.
Familiar with cloud platforms including AWS and Google Cloud.
Excellent communication skills and teamwork abilities.
"""
skills_1 = extractor.extract(cv_text_1)
print(f'\nCV: Data Scientist')
print(f'Skills extracted ({len(skills_1)}): {skills_1}')

# ── Demo 2: CV Asli Berantakan (Uji Coba Filter Link & Simbol) ──
cv_text_2 = """
Yehezkiel Situmorang
Email: yehezkiels.28022005@gmail.com | Phone: (+62)8111023149
Linkedin: linkedin.com/in/yehezkiel-situmorang-a127bb345/ | Github: github.com/KielSitum
GPA: 3.79 / 4.00 at Universitas Sumatera Utara

TECHNICAL SKILLS
• Programming Languages: C, C++, Java, Flutter.
• Web Development: HTML, CSS, JavaScript, PHP, MySQL, Laravel, Express.
"""
skills_2 = extractor.extract(cv_text_2)
print(f'\nCV: IT Student (Real Case Test)')
print(f'Skills extracted ({len(skills_2)}): {skills_2}')

# ── Demo 3: CV Marketing ──
cv_text_3 = """
Marketing professional with 5 years in digital marketing and content strategy.
Skills include SEO, social media marketing, and Google Analytics.
Experienced in email marketing, copywriting, and brand management.
Strong leadership and customer service background.
"""
skills_3 = extractor.extract(cv_text_3)
print(f'\nCV: Marketing')
print(f'Skills extracted ({len(skills_3)}): {skills_3}')

## Cell 16 — Ringkasan & Checklist Main Quest / Side Quest

In [ ]:
# Load training log
with open('training_log.json') as f:
    log = json.load(f)

best_epoch = max(log, key=lambda x: x['val_accuracy'])
final_val_acc = best_epoch['val_accuracy']
final_val_loss = best_epoch['val_loss']

print('=' * 65)
print('  KARISMA AI — MODEL 1: SKILL EXTRACTOR — RINGKASAN AKHIR')
print('=' * 65)
print()
print(f'  Best Val Accuracy : {final_val_acc:.4f} ({final_val_acc*100:.2f}%)')
print(f'  Best Val Loss     : {final_val_loss:.4f}')
print(f'  Target Akurasi    : {"TERCAPAI" if final_val_acc >= 0.85 else "❌ Belum"} (≥ 85%)')
print()
print('MAIN QUEST CHECKLIST:')
print('Model Deep Learning (TF Functional API) — DistilBERT + Keras head')
print('Custom Component — SkillExtractionCallback (Custom Callback)')
print('Custom Loss — MaskedSparseCategoricalCrossentropy')
print('Model tersimpan dalam format .keras (siap produksi)')
print('Kode inference tersedia (class SkillExtractor)')
print()
print('SIDE QUEST CHECKLIST (Model 1):')
print('TensorBoard — log disimpan di ./logs/model1_skill_extractor')
print('Akurasi ≥ 85%')
print('FastAPI — akan diimplementasikan terpisah (serve Model 1 & 2)')
print('tf.GradientTape — di Model 2 (Career Classifier)')
print('Custom Layer — di Model 2 (SkillProjectionLayer)')
print()
print('  OUTPUT FILES:')
print('karisma_skill_extractor_v1.keras')
print('karisma_skill_extractor_savedmodel/')
print('skill_vocab.json')
print('training_log.json')
print('training_history.png')
print('eda_skill_distribution.png')
print()
print('  NEXT STEP: Kirim output ["python", "sql", ...] ke Model 2!')
print('=' * 65)